# Prediction of Complications After Myocardial Infarction

AI course project, Amirkabir University of Technology. Dataset: MI Complications (UCI, id 579).

This notebook reads the real outputs produced by the pipeline (`outputs/metrics`, `outputs/figures`, `outputs/submission`) and displays them. Three outcomes are predicted at admission time: in-hospital death (`LET_IS`), chronic heart failure (`ZSN`) and atrial fibrillation (`FIBR_PREDS`).

**Usage:** if `python run_all.py` has already been run, execute the cells in order. Otherwise set `RUN_PIPELINE = True` in the third code cell to run the whole pipeline (requires `data/raw/MI.data`).

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "imblearn": "imbalanced-learn",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
    "shap": "shap",
    "matplotlib": "matplotlib",
}

missing_packages = [
    package
    for module, package in REQUIRED.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    print("Installing:", missing_packages)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing_packages]
    )

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

METRICS = ROOT / "outputs" / "metrics"
FIGURES = ROOT / "outputs" / "figures"
SUBMISSION = ROOT / "outputs" / "submission"
TARGETS = ["LET_IS", "ZSN", "FIBR_PREDS"]

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

In [ ]:
RUN_PIPELINE = False
FAST = False

if RUN_PIPELINE:
    command = [sys.executable, str(ROOT / "run_all.py")]
    if FAST:
        command.append("--fast")
    subprocess.run(command, cwd=ROOT, check=True)

In [ ]:
def read(name):
    path = METRICS / name
    if not path.exists():
        print(f"missing: {name}")
        return None
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        print(f"empty: {name}")
        return pd.DataFrame()


def show_image(name, width=900):
    path = FIGURES / name
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"missing: {name}")


def pm(frame, key, digits=3):
    return frame.apply(
        lambda r: f"{r[key + '_mean']:.{digits}f} ± {r[key + '_std']:.{digits}f}",
        axis=1,
    )

## 1. Dataset and data split

- 1700 patients. The ID column, the 12 outcome columns and the 9 forbidden columns of the admission scenario are removed from the inputs.
- 20% of the data (stratified on death, `random_state=42`) is locked as the test set and used only once.
- All preprocessing lives inside a `Pipeline` and is fitted only on the training part of each fold.

In [ ]:
from common import load_train, load_test

X_train, y_train, _ = load_train()
X_test, y_test, _ = load_test()

summary = pd.DataFrame(
    {
        "train_n": [len(y_train)] * 3,
        "test_n": [len(y_test)] * 3,
        "train_positives": [int(y_train[t].sum()) for t in TARGETS],
        "test_positives": [int(y_test[t].sum()) for t in TARGETS],
        "train_prevalence": [y_train[t].mean() for t in TARGETS],
        "test_prevalence": [y_test[t].mean() for t in TARGETS],
    },
    index=TARGETS,
)
print("selected input features:", X_train.shape[1])
summary

In [ ]:
missing = read("missingness_feature_selection.csv")
if missing is not None:
    key = [c for c in missing.columns if "missing" in c.lower() and "percent" in c.lower() or "rate" in c.lower()]
    display(missing.sort_values(key[0], ascending=False).head(12) if key else missing.head(12))

## 2. Cross-validation results (5-fold) for the mandatory models

In [ ]:
cv = read("cv_results_tuned.csv")

for target in TARGETS:
    sub = cv[cv["target"] == target]
    prevalence = sub["prevalence"].iloc[0]
    view = pd.DataFrame(
        {
            "model": sub["model"],
            "AUROC": pm(sub, "auroc"),
            "AUPRC": pm(sub, "auprc"),
            "AUPRC/pi": sub["auprc_ratio_mean"].round(2),
            "Brier": pm(sub, "brier", 4),
            "BSS": sub["bss_mean"].round(3),
            "final": sub["is_final"].map({True: "✓", False: ""}),
        }
    ).reset_index(drop=True)
    display(Markdown(f"### {target} — prevalence = {prevalence:.3f}, baseline Brier = {prevalence * (1 - prevalence):.4f}"))
    display(view)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, target in zip(axes, TARGETS):
    sub = cv[cv["target"] == target]
    ax.bar(sub["model"], sub["auprc_mean"], yerr=sub["auprc_std"], capsize=4)
    ax.axhline(sub["prevalence"].iloc[0], color="r", linestyle="--", label="prevalence")
    ax.set_title(f"AUPRC (CV): {target}")
    ax.tick_params(axis="x", rotation=30)
    ax.legend()
fig.tight_layout()
plt.show()

**Conclusion (cross-validation).** Numbers below are from our run (seed 42, 5-fold CV on the 1360 training patients).

- **Death (`LET_IS`, prevalence 0.160):** XGBoost reaches AUROC 0.898 ± 0.011 and AUPRC 0.698 ± 0.025 (4.4x the prevalence), clearly above logistic regression (AUPRC 0.645 ± 0.017) and the prevalence baseline.
- **Chronic heart failure (`ZSN`, 0.225):** the three tree models are equivalent (AUROC ≈ 0.72, AUPRC ≈ 0.51, about 2.3x the prevalence).
- **Atrial fibrillation (`FIBR_PREDS`, 0.099):** weak signal (AUROC ≈ 0.70, AUPRC 0.285, about 2.9x the prevalence).
- Differences between the tree models are within the fold standard deviation, so the choice of the "best" tree model is nearly arbitrary. The final model was picked by the AUPRC of the hyperparameter search.

## 3. Calibration (Platt is mandatory; isotonic and beta are bonus)

In [ ]:
cal = read("calibration_results.csv")
cal[["target", "final_model", "method", "brier", "bss", "log_loss", "ece", "cal_slope", "cal_intercept"]]

In [ ]:
show_image("calibration_before_after.png", 1100)
show_image("calibration_all_methods.png", 1100)

**Conclusion (calibration).** XGBoost already gives nearly calibrated probabilities (slope 1.01, ECE 0.017 for death). Platt scaling, which the assignment requires, made death slightly worse (Brier 0.0839 vs 0.0823, ECE 0.039 vs 0.017, slope 1.18) and was neutral for ZSN. The cross-fitted beta calibration was equal or better for all three outcomes; isotonic regression helped death and ZSN but hurt atrial fibrillation. The submission file uses Platt to follow the assignment. Set `MI_CALIBRATION=isotonic` to calibrate the final model with isotonic regression instead.

A calibration slope close to 1 and an intercept close to 0 mean the probabilities are trustworthy. A slope below 1 indicates overconfident (overfitted) probabilities.

## 4. Decision threshold selection

In [ ]:
thr = read("threshold_summary_oof.csv")
thr

**Reading the thresholds.** The 80%-sensitivity rule picks the largest threshold that still catches at least 80% of the positive cases; the cost-based rule uses `t* = C_FP / (C_FP + C_FN)`. For a 10:1 cost ratio this is 1/11 ≈ 0.091. Low thresholds mean many alerts and a low PPV; the default threshold 0.5 would detect almost nobody for rare outcomes.

Primary threshold: the largest threshold that reaches at least 80% sensitivity (on out-of-fold probabilities of the training data). Cost-based threshold: `t* = C_FP / (C_FP + C_FN)` with a cost ratio of 10:1 for death and 3:1 for the other two outcomes.

## 5. Final evaluation on the test set (used once)

In [ ]:
test = read("test_results.csv")
test[["target", "model", "prevalence", "auroc", "auprc", "auprc_ratio", "brier", "bss", "log_loss", "ece"]]

In [ ]:
checks = []
for target in TARGETS:
    sub = test[test["target"] == target]
    dummy = sub[sub["model_key"] == "dummy"].iloc[0]
    final = sub[sub["is_final"]].iloc[0]
    checks.append(
        {
            "target": target,
            "final_model": final["model_key"],
            "AUPRC": final["auprc"],
            "prevalence": final["prevalence"],
            "AUPRC > prevalence": bool(final["auprc"] > final["prevalence"]),
            "Brier": final["brier"],
            "baseline Brier": dummy["brier"],
            "Brier < baseline": bool(final["brier"] < dummy["brier"]),
        }
    )
pd.DataFrame(checks)

**Conclusion (test set).** On the locked test set (340 patients, about 54 deaths) death reaches AUROC 0.887 and AUPRC 0.692, consistent with cross-validation (0.898 and 0.698), so the model is not overfitted to the CV folds. For all three outcomes AUPRC is above the prevalence and Brier is below the baseline, but for atrial fibrillation the margin is negligible (BSS about 0.02) and logistic regression has a slightly lower test Brier (0.0895 vs 0.0907).

In [ ]:
thr_test = read("test_threshold_results.csv")
thr_test[["target", "strategy", "is_primary", "threshold", "tn", "fp", "fn", "tp", "sensitivity", "specificity", "ppv", "f1", "alerts_per_100"]]

In [ ]:
show_image("test_calibration_confusion.png", 1100)

In [ ]:
ci = read("test_bootstrap_ci.csv")
cmp_ = read("test_model_comparison.csv")
display(ci)
display(cmp_)

**Conclusion (uncertainty).** The 95% bootstrap interval for the test AUROC of death is [0.839, 0.930] and for AUPRC [0.570, 0.802]. XGBoost vs logistic regression: the AUPRC difference is +0.149 (CI [0.072, 0.226]) for death and +0.139 for ZSN, both excluding zero; the DeLong AUROC tests give p = 0.020 (death) and 0.044 (ZSN), which do not survive a Bonferroni correction for three tests (alpha = 0.0167). For atrial fibrillation there is no difference (p = 0.78).

In [ ]:
submission = pd.read_csv(SUBMISSION / "test_predictions.csv")
print(submission.shape)
submission.head()

## 6. Model interpretation with SHAP (death prediction)

In [ ]:
shap_importance = read("shap_importance_LET_IS.csv")
shap_importance.head(15)

In [ ]:
show_image("shap_summary_LET_IS.png", 900)
show_image("shap_bar_LET_IS.png", 800)

**Reading the SHAP plot.** Each dot is a patient; the horizontal position is the contribution of the feature to the predicted death risk, the colour is the feature value. SHAP describes what the model relies on; it is not a causal statement (see Question 4). Compare the top-ranked features with `shap_importance_LET_IS.csv` from your run.

In [ ]:
local = read("shap_local_patients.csv")
display(local)
show_image("shap_local_high_risk_death.png", 800)
show_image("shap_local_low_risk_survivor.png", 800)

## 7. The four mandatory questions

### Question 1: Data leakage (SMOTE/imputation before the split vs. inside the Pipeline)

In [ ]:
q1 = read("q1_leakage.csv")
view = q1[["target", "setting"]].copy()
view["AUROC"] = pm(q1, "auroc")
view["AUPRC"] = pm(q1, "auprc")
view

**Conclusion (Question 1).** Doing SMOTE before the split inflates AUROC from 0.878 to 0.986 for death (+0.108), from 0.708 to 0.945 for ZSN (+0.237) and from 0.710 to 0.989 for atrial fibrillation (+0.279). Setting B (everything inside the Pipeline) is the trustworthy number. AUPRC in setting A is partly inflated by the changed class balance, so AUROC is the cleanest measure of leakage. In this dataset SMOTE gives no benefit (B ≈ D) and imputation before CV leaks almost nothing (C ≈ D).

Setting B (SMOTE inside the Pipeline) is the trustworthy number. In setting A synthetic samples and their neighbours are spread across the training and validation folds, and evaluation happens on a manipulated class distribution.

### Question 2: Probabilities after 1:1 undersampling

In [ ]:
q2 = read("q2_undersampling.csv")
q2[["variant", "mean_predicted", "true_prevalence", "brier", "ece", "cal_slope", "cal_intercept"]]

In [ ]:
show_image("q2_undersampling_calibration.png", 650)

**What to look for (Question 2).** After 1:1 undersampling the model learns a prevalence of about 50%, so the mean predicted death probability is far above the true prevalence (0.16). The prior correction `p = βp_s / (βp_s − p_s + 1)` with β = n_pos/n_neg removes this shift; Platt scaling on held-out data corrects it too and also fixes the slope. Compare `mean_predicted` with `true_prevalence` in the table above.

### Question 3: Threshold with a 10:1 cost ratio

In [ ]:
q3 = read("q3_threshold_cost10.csv")
q3[["rule", "threshold", "alerts_per_100", "detected_per_100", "missed_per_100", "false_alarms_per_100", "sensitivity", "specificity", "expected_cost_per_patient"]]

**Reading Question 3.** With a cost ratio of 10:1 the optimal threshold for a calibrated probability is 1/11 ≈ 0.091. The table shows, per 100 patients, how many alerts are raised, how many deaths are detected and missed, and how many false alarms occur. The empirical minimum-cost threshold on the out-of-fold probabilities should lie close to the theoretical value when the probabilities are calibrated.

Theoretical threshold: `t* = 1/(1+10) ≈ 0.091`.

### Question 4: Prediction vs. causation

In [ ]:
q4 = read("q4_causality.csv")
q4

**Conclusion (Question 4).** A drug that ranks high in SHAP (for example ICU heparin) does not cause the outcome. Physicians prescribe according to the severity of the patient (confounding by indication), and in-hospital drugs are only recorded for patients who survived long enough to receive them (survivorship bias). The crude death rates with and without each drug are shown above; treat them as associations, not effects.

An association between a drug and death is not causal: prescribing is a physician decision that depends on how sick the patient is (confounding by indication), and in-hospital drugs are only recorded for patients who survived until that moment (survivorship bias).

## 8. Questions of the selected paper (Yavru & Yılmaz Gündüz, 2022)

In [ ]:
paper = read("paper_q2_yavru_always_negative.csv")
paper

In [ ]:
mean_acc = paper.loc[paper["target"] == "MEAN", "always_negative_accuracy"].iloc[0]
print(f"mean accuracy of an always-negative model over 12 outputs: {mean_acc:.4f}")
print("accuracy reported by the paper: 0.9209")

**Conclusion (paper question 2).** The always-negative model reaches 92.06% mean accuracy over the 12 outputs, essentially the 92.09% reported by the paper, while detecting no patient (sensitivity 0). Accuracy is therefore not an informative metric on this dataset.

An always-negative model with zero sensitivity reaches almost the same accuracy as the one reported in the paper. Accuracy is therefore not an appropriate metric here; AUPRC, calibration and sensitivity should be reported instead.

## 9. Bonus section

### 72-hour scenario vs. admission time

In [ ]:
read("bonus_scenario_72h.csv")

**Conclusion (72-hour scenario).** Adding pain recurrence and opioid/NSAID counts from days 1-3 does not help: death AUROC 0.893 ± 0.011 at 72 h vs 0.898 ± 0.011 at admission, AUPRC 0.695 vs 0.698. ZSN and atrial fibrillation change by less than 0.01 AUROC. These columns are recorded after admission, so they would be leakage in the admission scenario anyway.

### The 9 additional complications (including the AUROC confidence interval for supraventricular tachycardia)

In [ ]:
read("bonus_other_complications_cv.csv")

**Conclusion (other complications).** Supraventricular tachycardia (`PREDS_TAH`, only 16 positives in training) has AUROC 0.42 with a 95% interval of [0.30, 0.54], so it is not predictable from admission data. Ranking works for the other rare complications (AUROC 0.62-0.80), but the Brier skill score stays below 0.11, so the probabilities barely beat the prevalence. Test AUROC for outcomes with fewer than 15 positive test cases (for example rupture, 9 cases) is unreliable.

### Classifier Chain vs. independent models

In [ ]:
chain = read("bonus_classifier_chain.csv")
chain[chain["main_target"]][["target", "model", "auroc", "auprc", "brier"]]

**Conclusion (multi-label).** A logistic-regression classifier chain gives the same AUROC and AUPRC as independent models (death 0.869 vs 0.869, ZSN 0.649 vs 0.649), so the correlation between outcomes brings no measurable gain here.

### Consistency with myocardial rupture

In [ ]:
read("bonus_rupture_consistency.csv").T

**Conclusion (rupture consistency).** All 45 training patients with myocardial rupture died. The death model cannot see rupture (it is an outcome), but gives them a higher mean risk (0.346 vs 0.154). Only 2.1% of patients receive a predicted death risk below the predicted rupture risk. Because the two models are trained independently, the logical constraint P(death) ≥ P(rupture) is not guaranteed.

### Missing-value indicator

In [ ]:
display(read("bonus_missing_indicator.csv"))
display(read("bonus_missingness_signal.csv").head(10))

**Conclusion (missing indicator).** Adding missing-value indicators raises death AUPRC from 0.698 to 0.759 and AUROC from 0.898 to 0.916, with no gain for ZSN and no gain for atrial fibrillation. The table of the strongest associations shows why: when the heart-failure history (`ZSN_A`) is missing, about 91% of the patients died; when the ICU blood pressure is missing, only 1.8% died. This is *informative missingness*. It probably reflects the recording process (for example patients who die early have no history recorded) rather than biology, so it may not be usable in a real deployment. The main model is therefore trained without indicators.

### Missingness mechanism (is the missingness informative or an artifact?)

The table `bonus_missingness_only_model` trains a model that sees **only** the missing-value indicators, never a real feature value. An AUROC clearly above 0.5 means that the pattern of missing values alone carries information about the outcome. `bonus_missingness_blocks` lists groups of features that are missing together; a block that is missing mostly for patients who died points to a recording artifact.

In [ ]:
display(read("bonus_missingness_only_model.csv"))
display(read("bonus_missingness_blocks.csv"))
display(read("bonus_missingness_count_bins.csv"))

### Subgroups (sex and age above 70)

In [ ]:
read("bonus_subgroups.csv")

**Conclusion (subgroups).** Death discrimination is somewhat better in men (AUROC 0.905) than in women (0.867) and in patients up to 70 years (0.900) than above 70 (0.856). With a single threshold the operating point differs between groups: above 70 years sensitivity is 0.89 but specificity only 0.62. Mean predicted risk is close to the observed prevalence in every group. Confidence intervals for the subgroups were not computed.

### Repeated CV with different seeds and nested CV

In [ ]:
display(read("bonus_repeated_cv_summary.csv"))
display(read("bonus_nested_cv.csv"))

**Conclusion (reliability of the estimates).** Nested CV gives AUROC 0.891 ± 0.012 for death, 0.728 ± 0.042 for ZSN and 0.714 ± 0.048 for atrial fibrillation; the optimism from tuning on the same folds is only about 0.007 AUPRC for death, so the main CV table is trustworthy. Repeated CV with different seeds only measures the partition noise, not sampling error: the XGBoost vs LightGBM difference (+0.011 AUPRC) is smaller than the fold standard deviation (≈ 0.03) and should be treated as a tie.

### Decision curve analysis (DCA)

In [ ]:
show_image("bonus_decision_curves.png", 1100)

**Conclusion (decision curves).** For death the model has a higher net benefit than "treat all" from a threshold of about 10% upwards (net benefit 0.110 vs 0.066 at 10%, 0.097 vs 0.011 at 15%); below about 5% there is no advantage. For atrial fibrillation the net benefit is close to zero at all thresholds. The curves use the out-of-fold probabilities of the training set.

## 10. Summary

In [ ]:
final_view = test[test["is_final"]][["target", "model", "prevalence", "auroc", "auprc", "auprc_ratio", "brier", "bss"]].reset_index(drop=True)
final_view

- The final model of each outcome was selected by CV AUPRC and calibrated with Platt scaling.
- Test results were computed only once and thresholds were chosen without looking at the test set.
- Deliverable probabilities file: `outputs/submission/test_predictions.csv` (and the 12-outcome version in `test_predictions_all12.csv`).
- Full text report: `REPORT.md`; Persian LaTeX report: `report/report.tex` (see the last cell).

## 11. Persian LaTeX report

In [ ]:
subprocess.run([sys.executable, str(ROOT / "src" / "generate_latex.py")], cwd=ROOT, check=True)
print("Compile with XeLaTeX:  cd report  &&  xelatex report.tex  &&  xelatex report.tex")